### 추론 파이프라인 구현
- 파일 : n1-3_pipeline.ipynb
- 역할 : 이미지 한 장이 결과가 되기까지의 다섯 단계를 코드로 구현합니다. 사전학습 모델 로딩부터 전처리·추론·후처리, Baseline 측정까지 이후 모든 실습의 기본 흐름입니다.
- 기능 :
    1. MobileNetV2 사전학습 모델 로딩  
    2. 모델 파라미터 수 확인  
    3. Resize와 Normalize  
    4. 차원 변환: HWC에서 NCHW로  
    5. 전처리 함수로 묶기  
    6. 추론 실행  
    7. 후처리: Softmax와 Top-5 해석  
    8. 동영상 프레임 단위 추론 루프  
    9. 측정 함수: Warm-up 후 반복 측정  
    10. MobileNetV2 CPU Baseline 측정  


>> [준비] 실습 저장소 받기 (노트북 전용 셀)
<hr>

In [ ]:
#- 실습 저장소 받기: 노트북마다 런타임이 분리되므로 새 세션에서 한 번 실행
import os
if not os.path.exists("/content/npu-course"):
    !git clone -q https://github.com/npu-ondevice-ai/npu-course.git /content/npu-course
%cd /content/npu-course

# 확인 포인트 ------------------------------------------
# - 현재 위치 /content/npu-course 출력 체크

>> [코드 2-1] MobileNetV2 사전학습 모델 로딩 (n1-3_pipeline.ipynb, STEP 1)
<hr>

In [ ]:
#- 모듈로딩
import torch
from torchvision import models

#- ImageNet 사전학습 가중치로 MobileNetV2 로딩
weights = models.MobileNet_V2_Weights.IMAGENET1K_V1     # 모델 가중치
model = models.mobilenet_v2(weights=weights)            # 모델 생성
model.eval()                                            # 추론 모드 전환

#- 이 가중치가 요구하는 전처리 규격 출력
preprocess_meta = weights.transforms()                  # ImageNet 모델 전처리기 추출
print(preprocess_meta)

# 확인 포인트 -------------------------------------------------------------------------
# - crop_size=[224], mean, std 값 출력 체크 (모델의 입력 계약)

**실행 결과 예시**
```
ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)
```

>> [코드 2-2] 모델 파라미터 수 확인 (STEP 2)
<hr>

In [ ]:
#- 모든 파라미터 텐서의 원소 수 합산
n_params = sum(p.numel() for p in model.parameters())
print(f"MobileNetV2 파라미터 수: {n_params:,}")

# 확인 포인트 -------------------------------------------------------------------------
# - 약 350만(3.5e6) 규모 체크

**실행 결과 예시**
```
MobileNetV2 파라미터 수: 3,504,872
```

>> [코드 2-3] Resize와 Normalize (STEP 3)
<hr>

In [ ]:
#- 모듈로딩
import numpy as np
from PIL import Image

#- 이미지 로딩과 크기 변환 (모델 요구 크기 224×224)
IMG_FILE = "data/sample.jpg"
img = Image.open(IMG_FILE).convert("RGB")
img_resized = img.resize((224, 224))

#- 0~255 픽셀을 0~1로 바꾼 뒤 채널별 Normalize
#  (1) 1차 범위 정규화(Scaling) 0.0~1.0
x = np.asarray(img_resized).astype(np.float32) / 255.0

# (2) 채널별 표준화(Standardization)
mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
x = (x - mean) / std

print(f"Normalize 후 shape: {x.shape}, dtype: {x.dtype}")

# 확인 포인트 -------------------------------------------------------------------------
# - (224, 224, 3) float32 출력 체크 (아직 HWC 순서)

**실행 결과 예시**
```
Normalize 후 shape: (224, 224, 3), dtype: float32
```

>> [코드 2-4] 차원 변환: HWC에서 NCHW로 (STEP 4)
<hr>

In [ ]:
#- 채널 순서 변경과 배치 차원 추가
x = x.transpose(2, 0, 1)                      # HWC → CHW
x = np.expand_dims(x, axis=0)                 # 배치 차원 추가 → NCHW
input_tensor = torch.from_numpy(x)            # 텐서로 변환

print(f"최종 입력 shape: {tuple(input_tensor.shape)}")

# 확인 포인트 -------------------------------------------------------------------------
# - 출력 체크

**실행 결과 예시**
```
최종 입력 shape: (1, 3, 224, 224)
```

>> [코드 2-5] 전처리 함수로 묶기 (STEP 5)
<hr>

In [ ]:
#- STEP 3~4의 전처리를 함수 하나로 정리
def preprocess(img):
    img = img.convert("RGB").resize((224, 224))               # 이미지 크기 변경
    x = np.asarray(img).astype(np.float32) / 255.0            # 범위 스케일링
    x = (x - mean) / std
    x = x.transpose(2, 0, 1)[None]                            # HWC → NCHW
    return torch.from_numpy(x.astype(np.float32))             # ndarray → tensor

#- 함수 동작 확인
IMG_FILE = "data/sample.jpg"
x_check = preprocess(Image.open(IMG_FILE))
print(tuple(x_check.shape), x_check.dtype, f"{x_check.min():.2f} ~ {x_check.max():.2f}")

# 확인 포인트 -------------------------------------------------------------------------
# - 이미지 텐서 출력 체크

>> [코드 2-6] 추론 실행 (STEP 6)
<hr>

In [ ]:
#- 기울기 계산 없이 순전파 1회 (추론 전용)
with torch.no_grad():                              # 기울기 계산 비활성화(추론 전용)
    logits = model(input_tensor)                   # 추론 결과

print(f"출력 shape  : {tuple(logits.shape)}")
print(f"출력 값 범위: [{logits.min():.2f}, {logits.max():.2f}]")

# 확인 포인트 -------------------------------------------------------------------------
# - (1, 1000) 출력 체크: 클래스 1,000개의 점수(logit)
# - 값이 확률(0~1)이 아님을 확인: 해석은 후처리의 몫

**실행 결과 예시**
```
출력 shape: (1, 1000)
출력 값 범위: [음수, 양수]  (값은 입력 이미지에 따라 다름)
```

>> [코드 2-7] 후처리: Softmax와 Top-5 해석 (STEP 7)
<hr>

In [ ]:
#- 점수를 확률로 바꾸고 상위 5개 선택
probs = torch.softmax(logits, dim=1)[0]
top5 = torch.topk(probs, k=5)

#- 클래스 번호를 이름으로 바꿔 출력
categories = weights.meta["categories"]
for score, idx in zip(top5.values, top5.indices):
    print(f"{categories[idx]:30s} {score.item()*100:5.1f}%")

# 확인 포인트 -------------------------------------------------------------------------
# - 입력 이미지와 어울리는 클래스가 상위에 오는지 확인
# - 1위 확률(확신도) 수준 확인

>> [코드 2-8] 동영상 프레임 단위 추론 루프 (STEP 8)
<hr>

In [ ]:
#- 모듈로딩
import cv2

#- 동영상 열기와 프레임 반복 처리
VIDEO_FILE = "data/sample_video.mp4"
cap = cv2.VideoCapture( VIDEO_FILE)

frame_count = 0
while True:
    ret, frame = cap.read()                             # 프레임 1장 획득 (BGR)
    if not ret: break

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)  # BGR → RGB 변환
    x = preprocess(Image.fromarray(frame_rgb))          # 동일한 전처리

    with torch.no_grad():
          out = model(x)                                # 프레임 단위 추론
    frame_count += 1
cap.release()
print(f"처리한 프레임 수: {frame_count}")

# 확인 포인트 -------------------------------------------------------------------------
# - 처리한 프레임 수 = 영상의 총 프레임 수 체크
# - OpenCV의 BGR을 RGB로 바꾸는 줄이 필요한 이유 정리 (입출력 오류 재현과 연결)

>> [코드 2-9] 측정 함수: Warm-up 후 반복 측정 (STEP 9)
<hr>

In [ ]:
#- 모듈로딩
import time

#- Warm-up 뒤 n회 반복 측정해 평균·중앙값·p95 반환
def measure(fn, n_warmup=10, n_runs=100):
    for _ in range(n_warmup):                           # Warm-up: 측정에서 제외
        fn()
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        fn()
        times.append((time.perf_counter() - t0) * 1000)  # ms
    times = np.array(times)
    return times.mean(), np.median(times), np.percentile(times, 95), times.max()

# 확인 포인트 -------------------------------------------------------------------------
# - 함수 정의 셀: 출력 없음
# - Warm-up을 측정에서 빼는 이유 정리

>> [코드 2-10] MobileNetV2 CPU Baseline 측정 (STEP 10)
<hr>

In [ ]:
#- 고정 크기의 무작위 입력으로 CPU Baseline 측정
x = torch.randn(1, 3, 224, 224)
#- 기울기 계산을 끈 상태로 측정
with torch.inference_mode():
     mean_ms, med_ms, p95_ms, max_ms = measure(lambda: model(x))

print(f"Latency 평균 {mean_ms:.2f} ms / 중앙값 {med_ms:.2f} ms / p95 {p95_ms:.2f} ms")
print(f"FPS {1000/mean_ms:.1f}")

# 확인 포인트 -------------------------------------------------------------------------
# - 절댓값은 머신마다 다름: 평균과 p95의 차이(꼬리 지연) 확인
# - FPS = 1000 / 평균 Latency 관계 체크